# KAN-NNUE Training in Bullet

Train KAN (B-spline) and baseline (SCReLU) NNUE networks on the same data,
then compare loss curves.

**Runtime**: Use a GPU runtime (T4 or better). Go to Runtime > Change runtime type > GPU.

Architecture:
- **KAN**: `768 -> ft(128) -> KAN(256->128) -> KAN(128->1) -> sigmoid`
- **Baseline**: `768 -> ft(128) -> SCReLU -> 256->128 -> SCReLU -> 128->1 -> sigmoid`

## 1. Setup: Install Rust & Clone Repo

In [ ]:
%%bash
# Install Rust (if not already installed)
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
source $HOME/.cargo/env

# Clone the bullet fork with KAN support
if [ ! -d /content/bullet ]; then
    git clone https://github.com/y0sif/bullet.git /content/bullet
fi
cd /content/bullet
git pull origin main
git log --oneline -5

## 2. Download Training Data

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack

## 3. Build Both Examples

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

# Check CUDA availability
nvidia-smi || echo "WARNING: No GPU detected. Training will be very slow."
echo "---"
nvcc --version || echo "WARNING: nvcc not found. Will try CPU fallback."

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

# Build both examples in release mode
# Uses default features (cuda) if nvcc is available, falls back to cpu
if command -v nvcc &> /dev/null; then
    echo "Building with CUDA support..."
    cargo build --release --example kan_simple --example kan_baseline 2>&1
else
    echo "Building with CPU only (no CUDA toolkit found)..."
    cargo build --no-default-features --features cpu --release --example kan_simple --example kan_baseline 2>&1
fi
echo "Build complete!"

## 4. Train Baseline (SCReLU)

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

if command -v nvcc &> /dev/null; then
    cargo run --release --example kan_baseline 2>&1 | tee /content/baseline_log.txt
else
    cargo run --no-default-features --features cpu --release --example kan_baseline 2>&1 | tee /content/baseline_log.txt
fi

## 5. Train KAN

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet

if command -v nvcc &> /dev/null; then
    cargo run --release --example kan_simple 2>&1 | tee /content/kan_log.txt
else
    cargo run --no-default-features --features cpu --release --example kan_simple 2>&1 | tee /content/kan_log.txt
fi

## 6. Compare Loss Curves

In [ ]:
import re
import matplotlib.pyplot as plt

def parse_bullet_log(path):
    """Parse Bullet training log to extract superbatch loss values."""
    losses = []
    with open(path) as f:
        for line in f:
            # Bullet prints lines like: "Superbatch 1/40 ... loss 0.XXXX"
            m = re.search(r'Superbatch\s+(\d+).+?loss\s+([\d.]+)', line)
            if m:
                losses.append((int(m.group(1)), float(m.group(2))))
    return losses

baseline = parse_bullet_log('/content/baseline_log.txt')
kan = parse_bullet_log('/content/kan_log.txt')

if baseline and kan:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot([x[0] for x in baseline], [x[1] for x in baseline], label='Baseline (SCReLU)', linewidth=2)
    ax.plot([x[0] for x in kan], [x[1] for x in kan], label='KAN (B-spline)', linewidth=2)
    ax.set_xlabel('Superbatch')
    ax.set_ylabel('Loss')
    ax.set_title('KAN vs Baseline NNUE Training Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/kan_vs_baseline.png', dpi=150)
    plt.show()

    # Print final loss comparison
    baseline_final = baseline[-1][1]
    kan_final = kan[-1][1]
    improvement = (baseline_final - kan_final) / baseline_final * 100
    print(f'\nBaseline final loss: {baseline_final:.6f}')
    print(f'KAN final loss:      {kan_final:.6f}')
    print(f'Improvement:         {improvement:+.1f}%')
else:
    print('Could not parse training logs. Check the log format above.')
    if not baseline:
        print('  - baseline_log.txt: no loss values found')
    if not kan:
        print('  - kan_log.txt: no loss values found')

## 7. Save Results

Checkpoints are saved to `/content/bullet/checkpoints/`. Copy to Drive if needed:

In [ ]:
# Optional: copy checkpoints and logs to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/bullet/checkpoints /content/drive/MyDrive/kanue-bullet/
# !cp /content/kan_log.txt /content/baseline_log.txt /content/kan_vs_baseline.png /content/drive/MyDrive/kanue-bullet/